# SFT LoRA with Cut Tokenizer (Qwen3-4B-Thinking)

This notebook mirrors NVARC's tokenizer cutting approach, then runs LoRA fine-tuning and evaluation on the cut model.

## Configuration

In [1]:
import os
print(os.getcwd())
os.chdir("/data/coding/ARC")

/data/coding/ARC/ARChitects


In [ ]:
# Config
from dataclasses import dataclass
import os
import random
import torch

@dataclass
class Config:
    model_path: str = 'models'
    dataset_path: str = 'SDG/data/mini'  # HF dataset with messages, for tokenizer cutting
    chat_template_path: str = 'SDG/chat_template.j2'
    cut_output_dir: str = 'outputs/models_C'
    max_vocab_size: int = 16
    load_in_4bit: bool = True
    bnb_4bit_quant_type: str = 'nf4'
    bnb_4bit_compute_dtype: str = 'bfloat16'
    bnb_4bit_use_double_quant: bool = True

    sdg_dataset_paths: tuple = (
        'SDG/data/mini',
        'SDG/data/concept',
        # 'SDG/data/arc_training',
        'SDG/data/rearc',
    )
    use_streaming: bool = False
    shuffle_buffer: int = 10000
    sample_size: int = 0
    sample_ratio: float = 0.0
    sample_seed: int = 42

    eval_root: str = 'data'
    eval_split: str = 'evaluation'
    max_eval_tasks: int = 0  # 0 means all tasks

    max_seq_len: int = 1024

    lora_r: int = 64
    lora_alpha: int = 32
    lora_dropout: float = 0.05
    lora_target_modules: tuple = (
        'q_proj', 'k_proj', 'v_proj', 'o_proj',
        'gate_proj', 'up_proj', 'down_proj',
    )

    output_dir: str = 'outputs/arc_lora_sft_C'
    per_device_train_batch_size: int = 2
    gradient_accumulation_steps: int = 4
    learning_rate: float = 1e-4
    num_train_epochs: int = 3
    logging_steps: int = 50
    save_steps: int = 200

cfg = Config()

def set_seed(seed: int = 42):
    random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(cfg.sample_seed)
print('device:', 'cuda' if torch.cuda.is_available() else 'cpu')

device: cuda


## Preparation

In [ ]:
!kaggle datasets download -d sorokin/nvarc-synthetic-puzzles
!unzip -o nvarc-synthetic-puzzles.zip -d SDG/NVARCsynthetic

In [3]:
# Check whether special tokens exist in base/cut tokenizers
from transformers import AutoTokenizer

special_tokens = ["<|endoftext|>", "<|im_start|>", "<|im_end|>", "user", "assistant"]

print('--- base tokenizer (model_path) ---')
try:
    base_tok = AutoTokenizer.from_pretrained(
        cfg.model_path, use_fast=False, local_files_only=True
    )
    for t in special_tokens:
        print(f"{t}:", base_tok.convert_tokens_to_ids(t))
except Exception as e:
    print('load base tokenizer failed:', e)

print('--- cut tokenizer (cut_output_dir) ---')
try:
    cut_tok = AutoTokenizer.from_pretrained(cfg.cut_output_dir, use_fast=False)
    for t in special_tokens:
        print(f"{t}:", cut_tok.convert_tokens_to_ids(t))
except Exception as e:
    print('load cut tokenizer failed:', e)

/data/miniconda/envs/torch/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


--- base tokenizer (model_path) ---
<|endoftext|>: 151643
<|im_start|>: 151644
<|im_end|>: 151645
user: 872
assistant: 77091
--- cut tokenizer (cut_output_dir) ---
load cut tokenizer failed: Couldn't instantiate the backend tokenizer from one of: 
(1) a `tokenizers` library serialization file, 
(2) a slow tokenizer instance to convert or 
(3) an equivalent slow tokenizer class to instantiate and convert. 
You need to have sentencepiece or tiktoken installed to convert a slow tokenizer to a fast one.


In [4]:
# Cut tokenizer vocab based on message dataset (NVARC-style)
from transformers import AutoTokenizer
from datasets import load_from_disk
import json

tokenizer = AutoTokenizer.from_pretrained(cfg.model_path)
with open(cfg.chat_template_path, 'r', encoding='utf-8') as f:
    tokenizer.chat_template = f.read()

ds = load_from_disk(cfg.dataset_path)

def get_tokens(sample):
    text = tokenizer.apply_chat_template(sample['messages'], tokenize=False)
    tokens = tokenizer.encode(text + '<|endoftext|>')
    sample['tokens'] = tokens
    return sample

ds = ds.map(get_tokens)

all_tokens = []
for sample in ds:
    all_tokens.extend(sample['tokens'])
all_tokens = sorted(set(all_tokens))

token_strs = [tokenizer.convert_ids_to_tokens(t) for t in all_tokens]
print('token_strs:', token_strs)

expected = set(['<|im_start|>', '<|im_end|>', '<|endoftext|>', 'Ċ', 'user', 'assistant'] + [str(i) for i in range(10)])
if set(token_strs) != expected:
    print('WARNING: token set differs from expected')
    print('expected:', expected)

if cfg.max_vocab_size and len(all_tokens) > cfg.max_vocab_size:
    raise ValueError(f'token count {len(all_tokens)} exceeds max_vocab_size {cfg.max_vocab_size}')

mapping = []
new_vocab = {}
for token_id in all_tokens:
    tok = tokenizer.convert_ids_to_tokens(token_id)
    new_vocab[tok] = len(mapping)
    mapping.append(token_id)

print('vocab size:', len(new_vocab))

os.makedirs(cfg.cut_output_dir, exist_ok=True)
with open(os.path.join(cfg.cut_output_dir, 'mapping.json'), 'w', encoding='utf-8') as f:
    json.dump({'old_ids': mapping, 'tokens': list(new_vocab.keys())}, f, ensure_ascii=False, indent=2)

token_strs: ['0', '1', '2', '3', '4', '5', '6', '7', '8', '9', 'Ċ', 'user', 'assistant', '<|endoftext|>', '<|im_start|>', '<|im_end|>']
vocab size: 16


In [9]:
# Build & save a truly cut tokenizer + model using mapping.json
import json, os, torch
from pathlib import Path
from transformers import AutoTokenizer, AutoModelForCausalLM

map_path = Path(cfg.cut_output_dir) / "mapping.json"
with open(map_path, "r", encoding="utf-8") as f:
    mapping = json.load(f)
old_ids = mapping["old_ids"]
tokens = mapping["tokens"]
new_vocab = {tok: i for i, tok in enumerate(tokens)}

# 1) 复制 tokenizer.json 并替换 vocab/merges
tok_json_path = Path(cfg.cut_output_dir) / "tokenizer.json"
tok = AutoTokenizer.from_pretrained(cfg.model_path, use_fast=False)
tok.save_pretrained(cfg.cut_output_dir)  # 先把原版存一份，再改写
tok_json = json.loads(tok_json_path.read_text(encoding="utf-8"))
tok_json["model"]["vocab"] = new_vocab
tok_json["model"]["merges"] = []  # 只保留单字节/特殊token，不再使用原 merges
tok_json_path.write_text(json.dumps(tok_json, ensure_ascii=False, indent=2), encoding="utf-8")

# 2) 裁剪模型权重
model = AutoModelForCausalLM.from_pretrained(cfg.model_path, device_map="cpu")
emb = model.get_input_embeddings()
lm_head = model.get_output_embeddings()
with torch.no_grad():
    emb.weight = torch.nn.Parameter(emb.weight[old_ids].clone())
    if lm_head is not None:
        lm_head.weight = torch.nn.Parameter(lm_head.weight[old_ids].clone())
model.config.vocab_size = len(old_ids)
model.tie_weights()
model.save_pretrained(cfg.cut_output_dir)

# 3) 校验
print("cut vocab size:", len(new_vocab))
print("embedding size:", model.get_input_embeddings().weight.shape)


len(tokenizer)

Writing model shards: 100%|██████████| 1/1 [00:08<00:00,  8.94s/it]

cut vocab size: 16
embedding size: torch.Size([16, 2560])


151669

In [5]:
# Build SFT training samples from multiple SDG augmented datasets
from datasets import load_from_disk, concatenate_datasets, interleave_datasets
from torch.utils.data import IterableDataset

all_datasets = []
for path in cfg.sdg_dataset_paths:
    ds = load_from_disk(path)
    all_datasets.append(ds)
    print(path, ds)

if not all_datasets:
    raise ValueError('no datasets loaded')

if cfg.use_streaming:
    stream_sets = []
    for ds in all_datasets:
        stream_sets.append(ds.to_iterable_dataset())
    ds = interleave_datasets(stream_sets)
    if cfg.shuffle_buffer and cfg.shuffle_buffer > 0:
        ds = ds.shuffle(buffer_size=cfg.shuffle_buffer, seed=cfg.sample_seed)
    if cfg.sample_size and cfg.sample_size > 0:
        ds = ds.take(cfg.sample_size)
    elif cfg.sample_ratio and cfg.sample_ratio > 0:
        raise ValueError('sample_ratio is not supported with streaming; use sample_size')
else:
    if len(all_datasets) == 1:
        ds = all_datasets[0]
    else:
        ds = concatenate_datasets(all_datasets)
    if cfg.sample_ratio and cfg.sample_ratio > 0:
        n = int(len(ds) * cfg.sample_ratio)
        if n < 1:
            raise ValueError('sample_ratio too small for dataset size')
        ds = ds.shuffle(seed=cfg.sample_seed).select(range(n))
    elif cfg.sample_size and cfg.sample_size > 0:
        ds = ds.shuffle(seed=cfg.sample_seed).select(range(cfg.sample_size))

print('streaming:', cfg.use_streaming)
if not cfg.use_streaming:
    print('total training samples:', len(ds))

def iter_samples(dataset):
    for row in dataset:
        msgs = row["messages"]
        last_idx = max(i for i, m in enumerate(msgs) if m["role"] == "assistant")
        prompt_msgs = msgs[:last_idx]
        target = msgs[last_idx]["content"]

        parts = []
        for m in prompt_msgs:
            parts.append("<|im_start|>" + m["role"] + "\n" + m["content"] + "<|im_end|>")
        parts.append("<|im_start|>assistant\n")
        prompt = "".join(parts)
        yield {"prompt": prompt, "target": target}

class StreamSFTDataset(IterableDataset):
    def __init__(self, dataset, tokenizer, max_seq_len: int):
        self._dataset = dataset
        self._tokenizer = tokenizer
        self._max_seq_len = max_seq_len

    def __iter__(self):
        for item in iter_samples(self._dataset):
            prompt_ids = self._tokenizer(item["prompt"], add_special_tokens=False)["input_ids"]
            target_ids = self._tokenizer(item["target"], add_special_tokens=False)["input_ids"]

            eos = self._tokenizer.eos_token_id
            input_ids = prompt_ids + target_ids + ([eos] if eos is not None else [])
            labels = [-100] * len(prompt_ids) + target_ids + ([eos] if eos is not None else [])

            if len(input_ids) > self._max_seq_len:
                input_ids = input_ids[-self._max_seq_len :]
                labels = labels[-self._max_seq_len :]

            yield {
                "input_ids": torch.tensor(input_ids, dtype=torch.long),
                "labels": torch.tensor(labels, dtype=torch.long),
                "attention_mask": torch.ones(len(input_ids), dtype=torch.long),
            }

SDG/data/mini Dataset({
    features: ['puzzle_name', 'messages'],
    num_rows: 37632
})
SDG/data/concept Dataset({
    features: ['puzzle_name', 'messages'],
    num_rows: 40960
})
SDG/data/rearc Dataset({
    features: ['puzzle_name', 'messages'],
    num_rows: 102378
})
streaming: False
total training samples: 180970


In [6]:
# Load cut model and tokenizer
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

compute_dtype = getattr(torch, cfg.bnb_4bit_compute_dtype, torch.float16)
bnb_config = None
if cfg.load_in_4bit:
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=cfg.bnb_4bit_use_double_quant,
        bnb_4bit_quant_type=cfg.bnb_4bit_quant_type,
        bnb_4bit_compute_dtype=compute_dtype,
    )

tokenizer = AutoTokenizer.from_pretrained(cfg.model_path, use_fast=False)
with open(cfg.chat_template_path, 'r', encoding='utf-8') as f:
    tokenizer.chat_template = f.read()

if tokenizer.eos_token_id is None:
    eos_id = tokenizer.convert_tokens_to_ids('<|endoftext|>')
    if eos_id is not None and eos_id != tokenizer.unk_token_id:
        tokenizer.eos_token_id = eos_id

if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token if tokenizer.eos_token else '<|endoftext|>'

model_kwargs = dict(device_map='auto')
if cfg.load_in_4bit:
    model_kwargs.update(dict(quantization_config=bnb_config, torch_dtype=compute_dtype))
else:
    model_kwargs.update(dict(torch_dtype=getattr(torch, 'bfloat16', None)))

model = AutoModelForCausalLM.from_pretrained(
    cfg.model_path,
    **model_kwargs,
)
model.config.use_cache = False


`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 398/398 [00:02<00:00, 153.38it/s, Materializing param=model.norm.weight]                              


In [7]:
# Compute max_new_tokens based on 30x30 grid
from eval.core import GridCodec

codec = GridCodec()
max_grid = [[0 for _ in range(30)] for _ in range(30)]
reply_text = codec.grid_to_text(max_grid) + '<|im_end|>'
max_new_tokens = len(tokenizer.encode(reply_text)) + 1
print('max_new_tokens:', max_new_tokens)

max_new_tokens: 931


## Pre-evalution

In [13]:
%load_ext autoreload
%autoreload 2

In [9]:
# Baseline evaluation with cut model (before LoRA)
from eval.solvers import RawSolver
from eval.models import HuggingFaceBackendTextGenerator
from eval.core_C import GridCodec, ARCDataset, run_evaluation

def get_truth(task, context):
    tests = task.get('test') or []
    if len(tests) != 1:
        return None
    return tests[0].get('output')

def get_task_id(task, context):
    return task.get('task_id', context.get('index'))

codec = GridCodec()

# Preview a couple of eval tasks
eval_dataset = ARCDataset(root=cfg.eval_root, split=cfg.eval_split, max_tasks=cfg.max_eval_tasks)
for i, task in enumerate(eval_dataset):
    if i >= 5:
        break
    print('--- task', task.get('task_id', i))
    print('train len:', len(task.get('train', [])), 'test len:', len(task.get('test', [])))
    print('sample train input:', task.get('train', [{}])[0].get('input'))
    print('sample train output:', task.get('train', [{}])[0].get('output'))


--- task 00576224
train len: 2 test len: 1
sample train input: [[8, 6], [6, 4]]
sample train output: [[8, 6, 8, 6, 8, 6], [6, 4, 6, 4, 6, 4], [6, 8, 6, 8, 6, 8], [4, 6, 4, 6, 4, 6], [8, 6, 8, 6, 8, 6], [6, 4, 6, 4, 6, 4]]
--- task 009d5c81
train len: 5 test len: 1
sample train input: [[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 8, 8, 0, 8, 8, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 8, 8, 8, 0, 0, 0], [0, 0, 0, 0, 0, 8, 8, 8, 8, 0, 0, 0, 0, 0], [0, 0, 0, 0, 8, 8, 0, 8, 0, 0, 8, 8, 0, 0], [0, 0, 0, 0, 0, 0, 0, 8, 8, 8, 8, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 8, 0, 8, 0, 0, 0], [0, 0, 0, 0, 0, 0, 8, 8, 8, 0, 8, 8, 8, 0], [0, 0, 0, 0, 0, 0, 8, 0, 0, 0, 0, 0, 8, 0], [0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]]
sample train output: [[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 2, 2, 0, 2

In [10]:
baseline_model = HuggingFaceBackendTextGenerator(
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=max_new_tokens,
)
baseline_solver = RawSolver(model=baseline_model, codec=codec)
baseline_reports = run_evaluation(
    dataset=eval_dataset,
    solver=baseline_solver,
    get_truth=get_truth,
    get_task_id=get_task_id,
    output_dir=os.path.join(cfg.output_dir, 'reports_baseline'),
    model_id=cfg.cut_output_dir,
    model_key='baseline_cut',
    viz_failures=True,
    max_tasks=cfg.max_eval_tasks,
)

print(baseline_reports['summary'])

The following generation flags are not valid and may be ignored: ['temperature', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


RawSolver prompt for test #0:
<|im_start|>userĊ86Ċ64<|im_end|><|im_start|>assistantĊ868686Ċ646464Ċ686868Ċ464646Ċ868686Ċ646464<|im_end|>Ċ<|im_start|>userĊ79Ċ43<|im_end|><|im_start|>assistantĊ797979Ċ434343Ċ979797Ċ343434Ċ797979Ċ434343<|im_end|>Ċ<|im_start|>userĊ32Ċ78<|im_end|><|im_start|>assistantĊ

RawSolver output for test #0:
323232Ċ787878Ċ232323Ċ878787Ċ323232Ċ787878
The user has been sending a series of messages with a pattern. Each message starts with a number (like 86, 79, 32) followed by a number (like 64, 43, 78), and then the assistant responds with a pattern that repeats those numbers in a specific way.

Let me break down the pattern:

1. User message: "Ċ86Ċ64" (the Ċ is probably a character that's being used as a separator or to indicate the start of a number sequence)
2. Assistant response: "Ċ868686Ċ646464Ċ686868Ċ464646Ċ868686Ċ646464"

It seems like the assistant is generating a string that:
- Starts with Ċ followed by the first number repeated 3 times (868686)
- Then Ċ follow

## LoRA

In [ ]:
# Build PyTorch Dataset and collator
from ARChitects.sft_utils import ArcSFTDataset, collate_sft

if cfg.use_streaming:
    train_dataset = StreamSFTDataset(ds, tokenizer, max_seq_len=cfg.max_seq_len)
else:
    samples = list(iter_samples(ds))
    train_dataset = ArcSFTDataset(samples, tokenizer, max_seq_len=cfg.max_seq_len)

def collate_fn(batch):
    return collate_sft(batch, tokenizer=tokenizer)

In [ ]:
# Freeze base; train LoRA only
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

if cfg.load_in_4bit:
    model = prepare_model_for_kbit_training(model)

for p in model.parameters():
    p.requires_grad = False

lora_config = LoraConfig(
    r=cfg.lora_r,
    lora_alpha=cfg.lora_alpha,
    lora_dropout=cfg.lora_dropout,
    target_modules=list(cfg.lora_target_modules),
    bias='none',
    task_type='CAUSAL_LM',
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()


In [ ]:
# Train LoRA
from transformers import TrainingArguments, Trainer

bf16_ok = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
training_args = TrainingArguments(
    output_dir=cfg.output_dir,
    per_device_train_batch_size=cfg.per_device_train_batch_size,
    gradient_accumulation_steps=cfg.gradient_accumulation_steps,
    learning_rate=cfg.learning_rate,
    num_train_epochs=cfg.num_train_epochs,
    logging_steps=cfg.logging_steps,
    save_steps=cfg.save_steps,
    bf16=bf16_ok,
    fp16=torch.cuda.is_available() and not bf16_ok,
    report_to='none',
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    data_collator=collate_fn,
)

trainer.train()

In [ ]:
# Save LoRA adapter
adapter_dir = os.path.join(cfg.output_dir, 'lora_adapter')
os.makedirs(adapter_dir, exist_ok=True)
model.save_pretrained(adapter_dir)
tokenizer.save_pretrained(adapter_dir)
print('saved:', adapter_dir)

## Evalution

In [ ]:
# Evaluate with eval/ and write reports
eval_model = HuggingFaceBackendTextGenerator(
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=cfg.max_new_tokens,
)
solver = RawSolver(model=eval_model, codec=codec)

reports = run_evaluation(
    dataset=eval_dataset,
    solver=solver,
    get_truth=get_truth,
    get_task_id=get_task_id,
    output_dir=os.path.join(cfg.output_dir, 'reports'),
    model_id=cfg.cut_output_dir,
    model_key='lora_sft_cut',
    viz_failures=True,
)

print(reports['summary'])

In [ ]:
# Save Lora-ed model as a merged one
from peft import PeftModel

base = AutoModelForCausalLM.from_pretrained(cfg.cut_output_dir, device_map='auto')
lora = PeftModel.from_pretrained(base, os.path.join(cfg.output_dir, 'lora_adapter'))
merged = lora.merge_and_unload()
merged.save_pretrained(os.path.join(cfg.output_dir, 'merged'))
tokenizer.save_pretrained(os.path.join(cfg.output_dir, 'merged'))